**Revisão de código**

Olá, Pedro! Meu nome é Alan e estou feliz em revisar seu projeto hoje!

Quando eu identificar um erro pela primeira vez, apenas o apontarei. A ideia é que você tente identificá-lo e corrigi-lo por conta própria. Ao longo da revisão, também farei observações sobre possíveis melhorias no código e compartilharei comentários sobre seu entendimento do tema. Caso encontre dificuldades para resolver algum ponto, na próxima iteração fornecerei dicas mais específicas e exemplos práticos para te ajudar. Também estarei aberto a feedback e discussões sobre os assuntos abordados.

Você poderá encontrar meus comentários em caixas verdes, amarelas ou vermelhas, como estas:

<div class="alert alert-block alert-success">
<b>Comentário do revisor: </b> <a class="tocSkip"></a>
Sucesso. Tudo está correto.
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor: </b> <a class="tocSkip"></a>
Observações. Algumas recomendações.
</div>

<div class="alert alert-block alert-danger">
<b>Comentário do revisor: </b> <a class="tocSkip"></a>
O bloco requer algumas correções. O trabalho não pode ser aceito com os comentários vermelhos.
</div>

Você pode me responder usando isto:


<div class="alert alert-block alert-info">
<b>Resposta do aluno</b> <a class="tocSkip"></a>
</div>



<div class="alert alert-block alert-danger">
<b>Comentário geral do revisor: </b> <a class="tocSkip"></a>

Olá, Pedro! Obrigado pelo envio.

A estrutura está certa: divisão 60/20/20 justificada pelo tamanho da base, busca de profundidade na árvore de decisão e de número de árvores na floresta aleatória, comparação com um modelo de referência (69%) e uma escolha final que pesa acurácia e custo computacional.

O projeto ainda não pode ser aceito, por dois pontos:

1. **Erro de execução na regressão logística:** a célula gera `NameError`, e a acurácia de validação citada no texto não aparece em nenhuma saída.
2. **Modelos avaliados no teste:** o teste usa os modelos da última volta de cada laço (profundidade 10 e 100 árvores), e não os melhores encontrados na validação.

Comentei cada etapa abaixo.
</div>

<div class="alert alert-block alert-success">
<b>Comentário geral do revisor (iteração 2): </b> <a class="tocSkip"></a>

Olá, Pedro! Obrigado pelo reenvio. Os dois pontos que bloqueavam foram resolvidos.

A regressão logística agora executa sem erro (0,7512 na validação). Os modelos escolhidos são treinados de novo com os melhores hiperparâmetros antes do teste: a árvore com profundidade 9 dá 0,7947, e a floresta com 20 árvores e profundidade 10 dá 0,8025, ambas acima de 0,75. Você também acrescentou `stratify`, a busca de `max_depth` na floresta e o `DummyClassifier` no teste (0,6936).

O projeto está **aprovado**. Respondi na caixa do teste à sua pergunta sobre a floresta com 100 árvores, e deixei em amarelo dois textos que ficaram desatualizados. Parabéns pelo trabalho!
</div>

## Inicialização

In [1]:
# loading all necessary libraries
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier


## Lendo os dados

In [2]:
# reading the main df
df = pd.read_csv('/datasets/users_behavior.csv')


## Avaliação inicial

In [3]:
# initial assessment of the df
df.info()
print('\n-', len(df), 'linhas e', len(df.columns), 'colunas.')
print('- df já tratado.')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB

- 3214 linhas e 5 colunas.
- df já tratado.


In [4]:
print(df.head())


   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0


**Sobre o df**

Conjunto de dados já está tratado e se caracteriza por ter colunas apresentando colunas com o comportamento mensal do cliente em relação ao consumo de cada produto oferecido pelo plano com uma coluna final classificando em qual plano o cliente é cadastrado

<div class="alert alert-block alert-success">
<b>Comentário do revisor — leitura dos dados: </b> <a class="tocSkip"></a>

A base foi inspecionada com `info()` e `head()`: 3.214 linhas, sem ausentes e com tipos numéricos, o que permite ir direto à modelagem.
</div>

## Separando os dados em características e objetivos

In [5]:
features = df.drop(columns='is_ultra')
target = df['is_ultra']

## Dividindo os dados de origem

* Como não há conjunto de testes separado, e o conjunto de origem só tem aproximadamente 3k linhas, ele será dividido na proporção 60/20/20
* Uma divisão inicial de 80/20 com uma seguda divisão dos 80 em 75/25 para resultar em 60/20/20

In [6]:
# carve out the set (20%)
features_temp, features_test, target_temp, target_test = train_test_split(
    features, target, test_size=0.20, stratify=target, random_state=123
)

# split the rest into train (60% of total) and validation (20% of total)
features_train, features_valid, target_train, target_valid = train_test_split(
    features_temp, target_temp, test_size=0.25, stratify=target_temp, random_state=123
)

<div class="alert alert-block alert-success">
<b>Comentário do revisor — divisão dos dados: </b> <a class="tocSkip"></a>

Correto: 60/20/20 em duas etapas (80/20 e depois 75/25), com a mesma semente, e a explicação da proporção está escrita antes do código. Como sugestão, `stratify=target` mantém a mesma proporção de clientes Ultra (31%) nos três conjuntos.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

`stratify=target` foi adicionado a divisão dos dados.
</div>

## Investigando com os dados de validação em três modelos de classificação

### Decision Tree Classifier

In [7]:
# defining initial evaluation parameters
best_dtc_depth_valid = 0
best_dtc_accuracy_valid = 0


In [8]:
# structuring the model and validating it with up to 10 depths
for depth in range (1, 11):
    model_dtc = DecisionTreeClassifier(random_state=123, max_depth=depth)
    model_dtc.fit(features_train, target_train)
    
    # measuring model accuracy
    accuracy_valid_dtc = model_dtc.score(features_valid, target_valid)
    print('Para o depth', depth, ', a accuracy encontrada foi de', accuracy_valid_dtc)
    
    # selecting the best accuracy and the best depth
    if accuracy_valid_dtc > best_dtc_accuracy_valid:
        best_dtc_accuracy_valid = accuracy_valid_dtc
        best_dtc_depth_valid = depth
        
print('\nA melhor acurácia do modelo Decision Tree Classifier para o conjunto de validação foi de:', 
      best_dtc_accuracy_valid, 'para a profundidade de:', best_dtc_depth_valid, '.')


Para o depth 1 , a accuracy encontrada foi de 0.7558320373250389
Para o depth 2 , a accuracy encontrada foi de 0.7729393468118196
Para o depth 3 , a accuracy encontrada foi de 0.7853810264385692
Para o depth 4 , a accuracy encontrada foi de 0.7853810264385692
Para o depth 5 , a accuracy encontrada foi de 0.7838258164852255
Para o depth 6 , a accuracy encontrada foi de 0.7776049766718507
Para o depth 7 , a accuracy encontrada foi de 0.7884914463452566
Para o depth 8 , a accuracy encontrada foi de 0.7853810264385692
Para o depth 9 , a accuracy encontrada foi de 0.7931570762052877
Para o depth 10 , a accuracy encontrada foi de 0.7807153965785381

A melhor acurácia do modelo Decision Tree Classifier para o conjunto de validação foi de: 0.7931570762052877 para a profundidade de: 9 .


<div class="alert alert-block alert-success">
<b>Comentário do revisor — árvore de decisão: </b> <a class="tocSkip"></a>

O laço de profundidade de 1 a 10 escolhe a melhor pela validação: 0,7947 com profundidade 9. Como sugestão, mostrar a acurácia de cada profundidade ajudaria a ver se o ganho se estabiliza, como o texto afirma.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

A confirmação visual da melhor acurácia havia sido feita, mas esta ocultada com o `#`. Agora de forma explicita foi demostrado e melhorado o texto.
</div>

### Random Forest Classifier

In [9]:
# defining initial evaluation parameters
best_rfc_est_valid = 0
best_rfc_accuracy_valid = 0
best_rfc_depth = 0


In [10]:
# structuring the model and validating it with up to 100 n_estimators
for est in range(10, 101, 10):
    for depth in range(1, 11):
        model_rfc = RandomForestClassifier(random_state=123, n_estimators=est, max_depth=depth)
        model_rfc.fit(features_train, target_train)

        # measuring model accuracy
        accuracy_valid_rfc = model_rfc.score(features_valid, target_valid)
        print('Para o est de', est, 'com depth de', depth, ', a accuracy encontrada foi de', accuracy_valid_rfc)
    
        # selecting the best accuracy and the best est
        if accuracy_valid_rfc > best_rfc_accuracy_valid:
            best_rfc_accuracy_valid = accuracy_valid_rfc
            best_rfc_est_valid = est
            best_rfc_depth = depth


Para o est de 10 com depth de 1 , a accuracy encontrada foi de 0.7573872472783826
Para o est de 10 com depth de 2 , a accuracy encontrada foi de 0.7776049766718507
Para o est de 10 com depth de 3 , a accuracy encontrada foi de 0.7838258164852255
Para o est de 10 com depth de 4 , a accuracy encontrada foi de 0.7947122861586314
Para o est de 10 com depth de 5 , a accuracy encontrada foi de 0.7884914463452566
Para o est de 10 com depth de 6 , a accuracy encontrada foi de 0.7931570762052877
Para o est de 10 com depth de 7 , a accuracy encontrada foi de 0.7947122861586314
Para o est de 10 com depth de 8 , a accuracy encontrada foi de 0.7916018662519441
Para o est de 10 com depth de 9 , a accuracy encontrada foi de 0.7947122861586314
Para o est de 10 com depth de 10 , a accuracy encontrada foi de 0.7978227060653188
Para o est de 20 com depth de 1 , a accuracy encontrada foi de 0.776049766718507
Para o est de 20 com depth de 2 , a accuracy encontrada foi de 0.7713841368584758
Para o est de 20

In [11]:
print('A melhor acurácia do modelo Random Forest Classifier no conjunto de validação foi de', 
      best_rfc_accuracy_valid, 'para a quantidade de', best_rfc_est_valid, 'estimadores, utilizando', 
      best_rfc_depth, 'níveis de profundidade.')

A melhor acurácia do modelo Random Forest Classifier no conjunto de validação foi de 0.8040435458786936 para a quantidade de 20 estimadores, utilizando 10 níveis de profundidade.


<div class="alert alert-block alert-success">
<b>Comentário do revisor — floresta aleatória: </b> <a class="tocSkip"></a>

A busca de 10 a 100 árvores encontra 0,8025 com 60 estimadores. Como sugestão, testar também `max_depth` daria um ajuste mais completo, já que a profundidade foi o hiperparâmetro decisivo na árvore.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Pela aula eu entendi que o depth para este modelo se aplicaria apenas no caso de regressão, então não coloquei aqui. Adicionado o ajuste do hiperparâmetro para o modelo.

`max_depth` foi adicionado ao modelo e foi encontrada uma acurácia ainda melhor.
</div>

### Logistic Regression

In [12]:
# structuring the model and validating it
model_lr = LogisticRegression(random_state=123, solver='liblinear')
model_lr.fit(features_train, target_train)

# measuring model accuracy
accuracy_valid_lr = model_lr.score(features_valid, target_valid).round(4)

print('A acurácia do modelo de Regressão Logística no conjunto de validação foi de', accuracy_valid_lr)

A acurácia do modelo de Regressão Logística no conjunto de validação foi de 0.7512


<div class="alert alert-block alert-danger">
<b>Comentário do revisor — regressão logística: </b> <a class="tocSkip"></a>

Esta célula foi executada e gerou `NameError`: o `print` usa `accuracy_valid`, mas a variável criada se chama `accuracy_valid_lr`. Por isso a acurácia de validação nunca é exibida, e o 0,703 citado no texto abaixo não pode ser conferido. Corrija o nome e execute a célula de novo.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Para o modelo **Logistic Regression** a variável `accuracy_valid_lr` foi ajustada e o resultado apresentado foi atualizado.

Todos os resultados foram atualizados no resumo abaixo em função de ter sido adicionado o stratify no processo anterior da divisão do dados.
</div>

<div class="alert alert-block alert-success">
<b>Comentário do revisor — regressão logística (iteração 2): </b> <a class="tocSkip"></a>

Resolvido. A variável foi corrigida e a acurácia de validação (0,7512) agora aparece na saída.
</div>

**Sobre a validação dos três modelos**

**Decision Tree Classifier:**
Para este modelo, a melhor acurácia foi de 0,7932 para uma profundidade 9. A partir desta profundidade o modelo tende a não apresentar melhoras efetivas na acurácia e o custo computacional fica maior.

**Random Forest Classifier:**
Para este modelo, a melhor acurácia foi de 0,804 para uma quantidade de 20 estimadores, utilizando 10 níveis de profundidade. Da mesma forma que aconteceu com o Decision Tree Classifier, para uma quantidade de estimadores maior e varendo vários níveis de profundidade, o custo computacional aumenta mas não há melhora tão grande na acurácia.

**Logistic Regression:**
Para este modelo, a acurácia ficou no limite aceitável do projeto, com um valor de 0,7512. 


<div class="alert alert-block alert-warning">
<b>Comentário do revisor — conclusão da validação: </b> <a class="tocSkip"></a>

A leitura das duas primeiras partes está certa. Para a regressão logística, o número só vale depois de a célula acima rodar sem erro. Como sugestão, a afirmação de que a acurácia não melhora com mais profundidade ou mais árvores precisa de uma saída que mostre isso.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Para o modelo **Logistic Regression** como não tenho como demostrar a afirmação que tinha feito, eu retirei a afirmação.
</div>


<div class="alert alert-block alert-success">
<b>Comentário do revisor — conclusão da validação (iteração 2): </b> <a class="tocSkip"></a>

Resolvido. A afirmação sem saída que a sustentasse saiu, e os números foram atualizados. Agora as acurácias de cada profundidade estão impressas, e dá para ver que o ganho oscila depois da profundidade 3.
</div>

## Verificando a qualidade dos modelos usando o conjunto de teste

### Decision Tree Classifier

* novo treinamento com o melhor modelo encontrado na validação

In [13]:
# training the model using the validated hyperparameters
model_dtc = DecisionTreeClassifier(random_state=123, max_depth=9)
model_dtc.fit(features_train, target_train)


DecisionTreeClassifier(max_depth=9, random_state=123)

In [14]:

# applying the test dataset to the model
test_predictins_dtc = model_dtc.predict(features_test)

# meansuring the model's accuracy on the test dataset
accuracy_test_dtc = accuracy_score(target_test, test_predictins_dtc).round(4)

print('A acurácia do modelo Decicion Tree Classifier para os dados de teste foi de', 
      accuracy_test_dtc)



A acurácia do modelo Decicion Tree Classifier para os dados de teste foi de 0.7947


<div class="alert alert-block alert-danger">
<b>Comentário do revisor — modelos avaliados no teste: </b> <a class="tocSkip"></a>

`model_dtc` e `model_rfc` guardam o **último** modelo treinado dentro de cada laço: a árvore com profundidade 10 e a floresta com 100 árvores. Os modelos escolhidos na validação (profundidade 9 e 60 árvores) não foram guardados, então os resultados de teste (0,7792 e 0,8009) são de outros modelos. O erro é silencioso: não há mensagem, mas o modelo avaliado não é o escolhido.

Treine de novo cada modelo com o melhor hiperparâmetro encontrado e só então avalie no teste. As comparações entre validação e teste no texto passam a fazer sentido depois disso.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Realizado novo treinamento no modelo **Decision Tree Clasifier** com seu respectivo melhor hiperparâmetro e o resultado final melhorou.

Contudo quando fiz o mesmo procedimento para o **Random Forest Classifier** houve uma melhora em relação ao primeiro projeto, mas em relação a este projeto atual ele melhorou menos....? como isso é possível?
</div>


<div class="alert alert-block alert-success">
<b>Comentário do revisor — modelos avaliados no teste (iteração 2): </b> <a class="tocSkip"></a>

Resolvido. Árvore e floresta são treinadas de novo com os hiperparâmetros escolhidos na validação antes do teste.

Sobre a sua pergunta: o 0,8118 "sem novo treinamento" vem do último modelo do laço, com 100 árvores e profundidade 10, o mesmo que você treina de novo mais abaixo, e por isso os dois dão 0,8118. Na validação, 20 árvores ganharam por pouco; no teste, 100 árvores ganham por pouco. São 643 exemplos de teste, então a diferença entre 0,8025 e 0,8118 são 6 previsões. É variação de amostra, e não sinal de que a escolha estava errada. O modelo final continua sendo o escolhido na validação: trocá-lo porque foi melhor no teste faria do teste um segundo conjunto de validação, e o número deixaria de medir o desempenho em dados novos.
</div>

### Random Forest Classifier

* aplicando os dados de teste sem um novo treinamento

In [15]:
# applying the test dataset to the model
test_predictions_rfc = model_rfc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rfc = accuracy_score(target_test, test_predictions_rfc).round(4)

print('A acurácia do modelo Random Forest Classifier para os dados de teste foi de',
     accuracy_test_rfc)


A acurácia do modelo Random Forest Classifier para os dados de teste foi de 0.8118


* aplicando os dados de teste com um novo treinamento com o melhor modelo encontrado na validação est=20

In [16]:
# training the model using the validated hyperparameters
model_rfc = RandomForestClassifier(random_state=123, n_estimators=20, max_depth=10)
model_rfc.fit(features_train, target_train)

RandomForestClassifier(max_depth=10, n_estimators=20, random_state=123)

In [17]:

# applying the test dataset to the model
test_predictions_rfc = model_rfc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rfc = accuracy_score(target_test, test_predictions_rfc).round(4)

print('A acurácia do modelo Random Forest Classifier para os dados de teste foi de',
     accuracy_test_rfc)



A acurácia do modelo Random Forest Classifier para os dados de teste foi de 0.8025


* aplicando o modelo com o último hiperparametro est=100

In [18]:
# training the model using the validated hyperparameters
model_rfc = RandomForestClassifier(random_state=123, n_estimators=100, max_depth=10)
model_rfc.fit(features_train, target_train)

RandomForestClassifier(max_depth=10, random_state=123)

In [19]:
# applying the test dataset to the model
test_predictions_rfc = model_rfc.predict(features_test)

# measuring the model's accuracy on the test dataset
accuracy_test_rfc = accuracy_score(target_test, test_predictions_rfc).round(4)

print('A acurácia do modelo Random Forest Classifier para os dados de teste foi de',
     accuracy_test_rfc)



A acurácia do modelo Random Forest Classifier para os dados de teste foi de 0.8118


### Logistic Regression

In [20]:
# applying the test dataser to the model
test_predictions_lr = model_lr.predict(features_test)

# measuring the model's accuracy on the the dataset
accuracy_test_rl = accuracy_score(target_test, test_predictions_lr).round(4)

print('A acurácia do modelo Logístic Regression para os dados de teste foi de',
     accuracy_test_rl)


A acurácia do modelo Logístic Regression para os dados de teste foi de 0.7683


**Sobre os resultados com os dados de teste para os modelos**

**Decision Tree Classifier:**
O resultado para os dados de teste apresentou uma acurácia de 0,7947. Este valor é muito próximo do 0,7932 apresentado para os dados de validação e demonstra um bom e coerente desempenho do modelo.

**Random Forest Classifier:**
O resultado para os dados de teste apresentou uma acurácia de 0,8118 antes de retreinar o modelo com o melhor hiperparâmetro e 0,8025 após treinar com o melhor hiperparâmetro. Assim como foi para o modelo Decision Tree Classifier, o valor da acurácia para os dados de teste foi próximo do valor da acurácia para os dados da validação (0,804). Desta forma, este modelo também apresenta um desempenho bom e coerente.

**Logistic Regression**
O resultado para os dados de teste apresentou uma acurácia de 0,7683. Este valor é muito próximo do 0.7512 apresentado para os dados de validação e demonstra um desempenho acima do limite mínimo para o projeto, mas abaixo das outras possibilidades disponíveis, por isso não deve ser a escolha para seguir com este projeto.

Os modelos Decision Tree e Random Forest apresentaram uma acurácia muito próxima entre si para os dados de teste. Levando em conta que tando nos dados de teste quando nos dados de valiação o resultado da acurácia foi acima do limite aceitável do teste e os valores entre si foram muito próximos, em caso de escolha é plausível, para este caso, a decisão pelo modelo Decision Tree.

Além de consumir teoricamente mais dados computacionais, durante o treinamento este modelo consumiu mais tempo para dar a resposta. Pensando na escalabilidade do projeto, uma quantidade de dados maiores resultaria em resultados mais demorados para o modelo Random Forest, logo optar por um modelo que entregar o esperado em um tempo menor mesmo com uma quantidade de dados maior é a opção mais rasoável.

<div class="alert alert-block alert-warning">
<b>Comentário do revisor — conclusão do teste e escolha do modelo: </b> <a class="tocSkip"></a>

O raciocínio de escolha é bom: com acurácias próximas, preferir o modelo mais leve é uma decisão de negócio defensável. Os números, porém, precisam ser refeitos com os modelos corretos, e a escolha deve ser confirmada com eles. A floresta pode continuar à frente, e a diferença de custo precisa ser comparada com isso.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Os números foram atualizados e melhoraram (alguns de forma ainda confusa de forma que marquei um 1:1 pra entender). A escolha foi confirmada.
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor — conclusão do teste e escolha do modelo (iteração 2): </b> <a class="tocSkip"></a>

A escolha está confirmada com os números corretos, e o argumento de custo continua válido. Dois ajustes no texto: o 0,8118 não é o resultado "antes de retreinar com o melhor hiperparâmetro", e sim o do modelo com 100 árvores; o resultado a reportar é o do modelo escolhido, 0,8025. E a regressão logística agora passa do limite no teste (0,7683), então o motivo para descartá-la é ficar abaixo das outras duas, como você mesmo escreveu.
</div>

## Tirando a prova real dos modelos

* Avaliando como um modelo burro responderia seguindo os dados brutos

In [21]:
dumb_accuracy = df['is_ultra'].value_counts(
    normalize=True).round(2).reset_index().rename(columns={'index': 'is_ultra',
                                                          'is_ultra': 'percent'})
print(dumb_accuracy, '\n')
print('A acurácia de um modelo burro é de', dumb_accuracy.loc[0, 'percent'])

   is_ultra  percent
0         0     0.69
1         1     0.31 

A acurácia de um modelo burro é de 0.69


In [22]:
# applying dummy model
dummy = DummyClassifier(strategy='most_frequent', random_state=123)
dummy.fit(features_train, target_train)
dummy_accuracy = dummy.score(features_test, target_test)

print('A acurácia do modelo dummy foi de', dummy_accuracy)

A acurácia do modelo dummy foi de 0.6936236391912908


<div class="alert alert-block alert-success">
<b>Comentário do revisor — prova real: </b> <a class="tocSkip"></a>

A comparação com um modelo que sempre prevê a classe mais comum (69%) é a referência certa, e a conclusão a usa para descartar a regressão logística. Como sugestão, calcule essa acurácia no conjunto de teste, com `DummyClassifier`, para que a comparação use os mesmos dados dos modelos.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Modelo dummy tradicional foi adicionado e confirmada a comparação.
</div>

**Avaliando o resultado dos modelos**

Tomando o modelo burro com referência, pode-se considerar que os resultado para o modelo **Logistic Regression** não tem representatividade para este estudo. Tanto nos dados de validação quanto nos dados de teste este modelo previu aproximadamente o que o modelo burro preveria, desta forma não argrega conhecimento e pode ser desconsiderado.

Tanto o modelo **Decision Tree Classifier** quando o **Random Forest Classifier** apresentaram uma resposta aproximadamente 15% melhor que o modelo burro e sempre acima do limite do projeto que é 0,75 de acurácia. Desta forma, é possível entender que estes modelos são válidos para utilização no projeto e com o acrescimo de dados de teste e consequente reajustes nos hiperparâmetros pode-se obter um resultado ainda melhor no futuro.

<div class="alert alert-block alert-warning">
<b>Comentário do revisor — avaliação final: </b> <a class="tocSkip"></a>

A conclusão está bem amarrada à referência. Revise o número: de 0,69 para cerca de 0,80 são 11 pontos percentuais, o que corresponde a uma melhora relativa de cerca de 16%, e não 13%. Atualize depois de refazer o teste.
</div>

<div class="alert alert-block alert-info">
<b>Resposta do aluno:</b> <a class="tocSkip"></a>

Considerei a melhoria do 0,69 para 0,81 como 14,81% então aproximei para 15%
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor — avaliação final (iteração 2): </b> <a class="tocSkip"></a>

A comparação com o `DummyClassifier` no teste ficou certa. Revise o primeiro parágrafo: a regressão logística (0,7683) agora fica 7 pontos acima do modelo de referência (0,6936), então não "prevê aproximadamente o que o modelo burro preveria". Sobre a porcentagem: a melhora relativa se calcula sobre a referência, (0,8025 − 0,6936) / 0,6936 ≈ 16% para a floresta e cerca de 15% para a árvore. O 14,81% usa 0,81 como base.
</div>